In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.messages import HumanMessage, AIMessage
load_dotenv()

True

In [2]:
llm = ChatOpenAI(model="gpt-4o-mini")
llm.invoke('hi')

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 8, 'total_tokens': 17, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_8656e4f1df', 'id': 'chatcmpl-EnPRfP5z1tHE7n8IQnFFtbDMF5Udb', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None, 'normalized_finish_reason': 'stop'}, id='lc_run--01a208d6-4703-73ec-9cd8-0b9d67718429-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 9, 'total_tokens': 17, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'au

In [3]:
history = InMemoryChatMessageHistory()
history.messages

C:\Users\sanke\AppData\Local\Temp\ipykernel_31988\3041462964.py:1: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  history = InMemoryChatMessageHistory()


[]

In [4]:
# lets create the prompt

prompt = ChatPromptTemplate.from_messages([
    ("system", """\
        You are a professional email support agent.
        Classify complaints as: Billing / Technical / General.
        Priority: High (financial impact > $100 or urgent) / Medium / Low.
        Remember all customer details throughout the conversation."""),
    MessagesPlaceholder(variable_name="history"),  # ← past turns injected here
    ("human", "{input}"),                           # ← new message here
])


# step 2 LCEL/pipeline

chain = prompt | llm | StrOutputParser()

In [5]:
chain

ChatPromptTemplate(input_variables=['history', 'input'], input_types={'history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annotated[langchain_c

In [6]:
store = {}
def get_session_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

In [7]:
store

{}

In [8]:
agent = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)

c:\Users\sanke\AI Bootcamp\ai-env\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [9]:
agent

RunnableWithMessageHistory(bound=RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  history: RunnableBinding(bound=RunnableLambda(_enter_history), kwargs={}, config={'run_name': 'load_history'}, config_factories=[])
}), kwargs={}, config={'run_name': 'insert_history'}, config_factories=[])
| RunnableBinding(bound=RunnableLambda(_call_runnable_sync), kwargs={}, config={'run_name': 'check_sync_or_async'}, config_factories=[]), kwargs={}, config={'run_name': 'RunnableWithMessageHistory'}, config_factories=[]), kwargs={}, config={}, config_factories=[], get_session_history=<function get_session_history at 0x00000218F0DC9620>, input_messages_key='input', history_messages_key='history', history_factory_config=[ConfigurableFieldSpec(id='session_id', annotation=<class 'str'>, name='Session ID', description='Unique identifier for a session.', default='', is_shared=True, dependencies=None)])

In [10]:
cfg = {"configurable":{"session_id":"123sanket"}}

In [11]:
r1 = agent.invoke({"input": "I received a billing complaint from John about invoice #1042. He was charged $500 instead of $250."},
                            config=cfg)

In [12]:
store

{'123sanket': InMemoryChatMessageHistory(messages=[HumanMessage(content='I received a billing complaint from John about invoice #1042. He was charged $500 instead of $250.', additional_kwargs={}, response_metadata={}), AIMessage(content='Complaint Type: Billing  \nPriority: High  \nCustomer Details: John  \nInvoice Number: #1042  \nIssue: Charged $500 instead of $250.  ', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])])}

In [13]:
r2 = agent.invoke(
    {"input": "What category does his complaint fall under?"},
    config=cfg
)

In [14]:
r2


"John's complaint falls under the Billing category."

In [15]:
store

{'123sanket': InMemoryChatMessageHistory(messages=[HumanMessage(content='I received a billing complaint from John about invoice #1042. He was charged $500 instead of $250.', additional_kwargs={}, response_metadata={}), AIMessage(content='Complaint Type: Billing  \nPriority: High  \nCustomer Details: John  \nInvoice Number: #1042  \nIssue: Charged $500 instead of $250.  ', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What category does his complaint fall under?', additional_kwargs={}, response_metadata={}), AIMessage(content="John's complaint falls under the Billing category.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])])}

In [16]:
cfg = {"configurable":{"session_id":"aniket_321"}}
agent.invoke(
    {"input": "What category does his complaint fall under?"},
    config=cfg
)

'Please provide the specifics of the complaint so I can assist you in categorizing it accurately.'

In [17]:
store

{'123sanket': InMemoryChatMessageHistory(messages=[HumanMessage(content='I received a billing complaint from John about invoice #1042. He was charged $500 instead of $250.', additional_kwargs={}, response_metadata={}), AIMessage(content='Complaint Type: Billing  \nPriority: High  \nCustomer Details: John  \nInvoice Number: #1042  \nIssue: Charged $500 instead of $250.  ', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What category does his complaint fall under?', additional_kwargs={}, response_metadata={}), AIMessage(content="John's complaint falls under the Billing category.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]),
 'aniket_321': InMemoryChatMessageHistory(messages=[HumanMessage(content='What category does his complaint fall under?', additional_kwargs={}, response_metadata={}), AIMessage(content='Please provide the specifics of the complaint so I can assist you in categorizing it a